# Week 4: Machine Learning on Databricks - Spark & Classification

## Learning Objectives

By the end of this session, you will be able to:

- Build and evaluate classification models using Spark MLlib (Logistic Regression, Random Forest, Gradient Boosted Trees)
- Handle imbalanced datasets using class weights and sampling strategies
- Implement cross-validation and hyperparameter tuning with CrossValidator
- Evaluate classification models using precision, recall, F1-score, and AUC-ROC metrics
- Build an end-to-end classification pipeline for customer churn prediction

## Prerequisites

- Completion of Week 3 (Spark regression fundamentals)
- Understanding of classification concepts from pre-class videos
- Familiarity with Databricks workspace and Spark DataFrames

## Session Overview

This week builds on your Week 3 Spark regression skills to tackle **classification problems** - predicting categorical outcomes rather than continuous values. You'll learn how to handle the unique challenges of classification, especially when dealing with **imbalanced datasets** where one class significantly outnumbers others (like customer churn scenarios).

**Real-World Context**: Customer churn prediction is critical for subscription-based businesses. Identifying customers likely to leave allows companies to take proactive retention actions. However, churn is typically rare (5-20% of customers), creating an imbalanced dataset challenge that requires specialized techniques.

**Topics Covered**:
1. Classification fundamentals with Logistic Regression
2. Handling imbalanced data (class weights, sampling strategies)
3. Advanced ensemble models (Random Forest, Gradient Boosted Trees)
4. Cross-validation and hyperparameter tuning

## Environment Setup

**Platform**: Azure Databricks Runtime 17.3 LTS ML

**Cluster Configuration**:
- Runtime: 17.3 LTS ML (includes Spark 4.0.0, Python 3.12.3)
- Cluster Type: Shared (Unity Catalog) - Provides secure multi-user isolation
- Node type: Standard_DS4_v2 (8 cores, 28GB RAM) or equivalent
- Autoscaling: 2-8 workers
- Fair Scheduler enabled for multi-user access

**Important Notes**:
- Shared clusters restrict `spark.sparkContext` access for security (by design)
- All operations use DataFrame API (RDD operations not supported on Shared clusters)
- This is the recommended configuration for secure multi-user environments

**Pre-installed Libraries** (included in Databricks ML runtime 17.3):
- PySpark 4.0.0 (Spark MLlib for machine learning)
- pandas 2.2.3 (data manipulation)
- matplotlib, seaborn (visualization)
- numpy 2.1.3 (numerical operations)
- PyTorch 2.7.0 (deep learning, if needed)

**No additional installations required** - Databricks ML runtime includes all necessary packages.

**Dataset**: IBM Telco Customer Churn dataset (~7,000 rows, ~27% churn rate)
- Source: IBM GitHub (https://github.com/IBM/telco-customer-churn-on-icp4d)
- Downloaded to DBFS: `/dbfs/shared_datasets/week_04_datasets/telecom_churn.csv`
- Features: Demographics, services (phone, internet), account info, charges
- Target: `Churn` (binary: "Yes" = churned, "No" = stayed)

In [ ]:
# Verify environment setup
# This cell confirms all required libraries are available

import pyspark
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display versions
print(f"PySpark version: {pyspark.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print("\n✅ Environment setup complete!")

In [ ]:
# Import all required libraries for classification
# These imports cover data loading, transformation, modeling, and evaluation

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Feature engineering and transformation
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler

# Classification models
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier

# Pipeline and tuning
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Evaluation metrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ All libraries imported successfully!")

## Section 0.5: Dataset Setup

**INSTRUCTOR/FIRST USER: Run this section ONCE before class starts** (5 minutes before students arrive)

This will download the UCI Telecom Churn dataset to the shared Unity Catalog Volume so all students can access it.

In [ ]:
# DATASET SETUP - Run this ONCE before class
# Downloads IBM Telco Customer Churn Dataset to shared DBFS storage

# DBFS path for shared datasets (works on all cluster types)
SHARED_DATA_PATH = "/dbfs/shared_datasets"
WEEK_04_DATASET = f"{SHARED_DATA_PATH}/week_04_datasets/telecom_churn.csv"

print("🔧 Week 4 Dataset Setup")
print("="*70)

# FORCE RE-DOWNLOAD: Remove old dataset to ensure IBM Telco is loaded
try:
    dbutils.fs.rm(f"{SHARED_DATA_PATH}/week_04_datasets/", recurse=True)
    print("🗑️  Removed old dataset directory (forcing fresh download)")
except Exception:
    print("📁 No existing dataset directory found (this is first setup)")

# Download IBM Telco Churn dataset from GitHub
print("\n📥 Downloading IBM Telco Customer Churn Dataset...")
print("   Source: IBM GitHub (official public dataset)")
print("   Size: ~7,000 rows, 21 features")
print("   Time: ~5-10 seconds")
print("")

try:
    import pandas as pd
    
    # Download from IBM GitHub repository
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    print(f"   ⏳ Downloading from: {url}")
    
    churn_pandas = pd.read_csv(url)
    print(f"   ✅ Downloaded: {len(churn_pandas):,} rows, {len(churn_pandas.columns)} columns")
    
    # Data cleaning: Fix TotalCharges column (contains empty strings)
    print("   ⏳ Cleaning data (fixing TotalCharges empty values)...")
    churn_pandas['TotalCharges'] = pd.to_numeric(churn_pandas['TotalCharges'], errors='coerce')
    
    # Fill null TotalCharges with 0 (new customers with tenure=0)
    churn_pandas['TotalCharges'].fillna(0, inplace=True)
    
    # Convert to Spark DataFrame
    churn_spark = spark.createDataFrame(churn_pandas)
    
    # Save to DBFS
    dbutils.fs.mkdirs(f"{SHARED_DATA_PATH}/week_04_datasets/")
    churn_spark.write.mode("overwrite").option("header", "true").csv(WEEK_04_DATASET)
    print(f"   ✅ Saved to shared DBFS: {WEEK_04_DATASET}")
    
    # Verify and cache
    verify_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(WEEK_04_DATASET)
    verify_count = verify_df.count()
    verify_cols = len(verify_df.columns)
    
    # Check for IBM dataset columns to confirm correct dataset
    schema_cols = [field.name for field in verify_df.schema.fields]
    if 'tenure' in schema_cols and 'Contract' in schema_cols:
        print(f"   ✅ Verified IBM Telco dataset: {verify_count:,} rows, {verify_cols} columns")
        print(f"   ✅ Confirmed columns: tenure, Contract, MonthlyCharges, TotalCharges")
    else:
        print(f"   ⚠️  WARNING: Dataset may be incorrect. Columns: {schema_cols[:5]}")
    
    verify_df.cache()
    verify_df.count()  # Trigger caching
    print(f"   ✅ Cached on cluster for fast student access")
    
    print("")
    print("="*70)
    print("🎉 Setup Complete! Students can now proceed.")
    print("="*70)
    
except Exception as e:
    print(f"\n❌ Setup failed: {str(e)}")
    print("\nTroubleshooting:")
    print("  1. Check internet connectivity")
    print("  2. Verify DBFS permissions (DS_Academy group)")
    print("  3. Ensure DBFS path is accessible: /dbfs/shared_datasets")
    print("  4. GitHub URL may have changed - check: https://github.com/IBM/telco-customer-churn-on-icp4d")
    raise

## Real-World Context: Customer Churn Prediction

### The Business Problem

**Scenario**: You're a data scientist at a major telecom company with millions of subscribers. Customer acquisition costs are high (\$300-500 per customer through marketing, sales, and activation), but customer churn rates hover around 15% annually. **Retaining existing customers is 5-7x cheaper than acquiring new ones.**

**Your Mission**: Build a machine learning model to predict which customers are likely to churn in the next 30 days, so the retention team can proactively offer personalized incentives (discounts, upgraded plans, premium support) to keep them.

### The Classification Challenge

Unlike Week 3's regression problem (predicting continuous taxi fares), **classification predicts discrete categories**:
- **Binary classification**: Two classes (e.g., churn vs. no churn, fraud vs. legitimate)
- **Multiclass classification**: Multiple classes (e.g., low/medium/high risk, product categories)

**Key Difference from Regression**:
- Regression: Predicts a number (fare amount, temperature, stock price)
- Classification: Predicts a category (churn/not churn, spam/not spam)

### The Imbalanced Data Problem

In most real-world classification scenarios, **classes are imbalanced**:
- **Churn**: 15% of customers (minority class) vs. 85% retained (majority class)
- **Fraud detection**: <1% fraudulent transactions vs. >99% legitimate
- **Disease diagnosis**: Rare diseases affect <5% of population

**Why this matters**: A naive model could achieve 85% accuracy by always predicting "no churn" - but this is useless for business! We need to **correctly identify the minority class** (churners) while minimizing false alarms.

### Success Metrics for This Session

By the end of this notebook, you'll build a model that:
1. **Identifies 70%+ of actual churners** (high recall on minority class)
2. **Maintains 60%+ precision** (avoiding too many false alarms)
3. **Achieves AUC-ROC > 0.80** (strong discriminative ability)
4. **Scales to millions of customers** (efficient Spark implementation)

Let's start by loading and exploring our telecom churn dataset.

In [ ]:
# Load the telecom customer churn dataset from DBFS
# Dataset downloaded in Section 0.5 by instructor

WEEK_04_DATASET = "/dbfs/shared_datasets/week_04_datasets/telecom_churn.csv"

# Load dataset from DBFS
churn_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(WEEK_04_DATASET)

# Display basic dataset information
print(f"Total customers: {churn_df.count():,}")
print(f"Total features: {len(churn_df.columns)}")
print(f"\nDataset schema:")
churn_df.printSchema()

# Show first few rows
print("\nSample data:")
churn_df.show(5, truncate=False)

---

## Topic 1: Classification Fundamentals with Logistic Regression

### Understanding Binary Classification

**Binary classification** is the task of categorizing data points into one of two classes. In our churn prediction scenario:
- **Class 0 (Negative)**: Customer stayed (no churn)
- **Class 1 (Positive)**: Customer churned (left the service)

**How it differs from regression**:
```python
# Regression (Week 3): Predict continuous value
fare_amount = model.predict(features)  # Returns: 45.67 (dollars)

# Classification (Week 4): Predict discrete class
churn_probability = model.predict(features)  # Returns: 0.78 (78% chance of churn)
churn_label = 1 if churn_probability > 0.5 else 0  # Binary decision
```

### Logistic Regression for Classification

Despite its name, **Logistic Regression is a classification algorithm**, not regression. It models the probability that an observation belongs to a particular class.

**Key concepts**:
1. **Logistic (Sigmoid) Function**: Transforms linear predictions into probabilities (0 to 1)
   - Formula: `P(y=1) = 1 / (1 + e^(-z))` where `z = w₁x₁ + w₂x₂ + ... + b`
   - Output always between 0 and 1, interpretable as probability

2. **Decision Boundary**: Threshold for converting probability to class label
   - Default: 0.5 (if P(churn) > 0.5, predict churn)
   - Can be adjusted based on business needs (more on this in Topic 2)

3. **Sparse vs. Dense Features**:
   - Logistic Regression handles both numerical and categorical features
   - Categorical features are typically one-hot encoded
   - Result is often high-dimensional sparse feature vectors

**When to use Logistic Regression**:
- ✅ Fast training and prediction (scales well to large datasets)
- ✅ Interpretable coefficients (understand feature importance)
- ✅ Works well when classes are linearly separable
- ✅ Good baseline model before trying complex algorithms
- ❌ Limited capacity for complex non-linear relationships

### Classification Metrics (Introduction)

Unlike regression (RMSE, R²), classification uses different metrics:

| Metric | What it measures | When to prioritize |
|--------|------------------|-------------------|
| **Accuracy** | Overall correctness | Balanced datasets |
| **Precision** | Of predicted churners, how many actually churned? | When false alarms are costly |
| **Recall** | Of actual churners, how many did we catch? | When missing positives is costly |
| **F1-Score** | Harmonic mean of precision and recall | Balance between precision/recall |
| **AUC-ROC** | Model's discriminative ability | Overall model quality |

**For churn prediction**: We prioritize **recall** (catching churners) while maintaining acceptable **precision** (not flagging too many false alarms).

In [ ]:
# Demo 1.1: Build a simple Logistic Regression classifier for churn prediction
# This demo shows the complete workflow from data prep to model evaluation

# Step 1: Prepare features and label
# First, let's check class distribution to understand the imbalance
print("Class distribution:")
churn_df.groupBy("Churn").count().show()

# Calculate churn rate
# Note: IBM Telco dataset has "Yes"/"No" strings for Churn
total_count = churn_df.count()
churn_count = churn_df.filter(F.col("Churn") == "Yes").count()
churn_rate = (churn_count / total_count) * 100
print(f"\nChurn rate: {churn_rate:.2f}%")

# Step 2: Select features for our baseline model
# For simplicity, we'll start with numerical features only
# (We'll add categorical features in the lab)
feature_cols = [
    "tenure",           # Months with company
    "MonthlyCharges",   # Monthly bill amount
    "TotalCharges"      # Total amount charged over customer lifetime
]

# Step 3: Convert Churn label from "Yes"/"No" to 0/1
# Use StringIndexer to convert categorical strings to numeric indices
label_indexer = StringIndexer(inputCol="Churn", outputCol="label")
churn_df_indexed = label_indexer.fit(churn_df).transform(churn_df)

# Step 4: Assemble features into a vector (required by Spark ML)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
feature_df = assembler.transform(churn_df_indexed)

# Step 5: Split into training and test sets (80/20 split)
train_data, test_data = feature_df.randomSplit([0.8, 0.2], seed=42)
print(f"\nTraining samples: {train_data.count():,}")
print(f"Test samples: {test_data.count():,}")

# Step 6: Train Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,           # Maximum iterations for convergence
    regParam=0.01         # Regularization parameter (prevents overfitting)
)

# Fit the model on training data
lr_model = lr.fit(train_data)
print("\n✅ Logistic Regression model trained successfully!")

# Step 7: Make predictions on test data
predictions = lr_model.transform(test_data)

# Display sample predictions
print("\nSample predictions (showing actual vs predicted):")
predictions.select("label", "prediction", "probability").show(10, truncate=False)

# Step 8: Evaluate model performance
# Binary classification evaluator for AUC-ROC
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator_auc.evaluate(predictions)

# Multiclass evaluator for accuracy, precision, recall, F1
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = evaluator_acc.evaluate(predictions)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
precision = evaluator_precision.evaluate(predictions)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
recall = evaluator_recall.evaluate(predictions)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)
f1 = evaluator_f1.evaluate(predictions)

# Display all metrics
print("\n" + "="*60)
print("BASELINE LOGISTIC REGRESSION PERFORMANCE")
print("="*60)
print(f"AUC-ROC:   {auc:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("="*60)

# Interpretation
print("\n📊 What these metrics tell us:")
print(f"- AUC-ROC ({auc:.4f}): Model can discriminate between churners/non-churners")
print(f"- Accuracy ({accuracy:.4f}): {accuracy*100:.1f}% of predictions are correct")
print(f"- Precision ({precision:.4f}): {precision*100:.1f}% of predicted churners actually churned")
print(f"- Recall ({recall:.4f}): We caught {recall*100:.1f}% of actual churners")
print(f"- F1-Score ({f1:.4f}): Balanced measure of precision and recall")

# Save churn_df_indexed for use in later demos
churn_df_labeled = churn_df_indexed

### Lab 1.1: Enhance the Baseline Model with Categorical Features

**Objective**: Improve the baseline Logistic Regression model by incorporating categorical features (e.g., `Contract`, `InternetService`, `PaymentMethod`) that likely influence churn behavior.

**Background**: Our baseline model only used 3 numerical features. However, the dataset contains rich categorical information:
- **Contract type**: Month-to-month customers may churn more easily than those with 1-year or 2-year contracts
- **Internet service**: Fiber optic vs DSL vs no internet may correlate with churn
- **Payment method**: Automatic payments vs manual may indicate customer commitment

**Your Task**: Build an enhanced Logistic Regression model that includes both numerical and categorical features.

**Step-by-Step Instructions**:

1. **Select additional features**:
   - Numerical: `tenure`, `MonthlyCharges`, `TotalCharges` (from baseline)
   - Categorical: `Contract`, `InternetService`, `PaymentMethod`, `OnlineSecurity`, `TechSupport`

2. **Handle categorical features**:
   - Use `StringIndexer` to convert categorical strings to indices
   - Use `OneHotEncoder` to create binary feature vectors
   - This prevents the model from assuming ordinal relationships (e.g., "Month-to-month" ≠ 1, "One year" ≠ 2)

3. **Build a Pipeline**:
   - Pipelines chain multiple transformations together
   - Makes code cleaner and prevents data leakage
   - Structure: StringIndexers → OneHotEncoders → VectorAssembler → LogisticRegression

4. **Train and evaluate**:
   - Use the same train/test split from the demo (80/20, seed=42)
   - Calculate all 5 metrics: AUC-ROC, Accuracy, Precision, Recall, F1-Score
   - Compare to baseline performance

**Expected Outcome**: Your enhanced model should achieve:
- AUC-ROC: > 0.80 (vs baseline ~0.75)
- Recall: > 0.60 (catching more churners)
- Precision: > 0.55 (maintaining reasonable accuracy on positive predictions)

**Hints**:
- Use `.stages` to add multiple transformers to a Pipeline
- Remember to use `handleInvalid="keep"` in StringIndexer to handle unseen categories
- VectorAssembler should combine both numerical features AND one-hot encoded categorical features

**Resources**:
- [Spark Pipeline documentation](https://spark.apache.org/docs/latest/ml-pipeline.html)
- [StringIndexer](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html)
- [OneHotEncoder](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.OneHotEncoder.html)

In [ ]:
# Lab 1.1: SOLUTION - Enhanced Logistic Regression with categorical features

# Step 1: Define feature sets
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]

categorical_features = ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport"]

# Step 2: Create StringIndexers for each categorical feature
string_indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in categorical_features
]

# Step 3: Create OneHotEncoders for each indexed categorical feature
one_hot_encoders = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    for col in categorical_features
]

# Step 4: Prepare feature column names for VectorAssembler
# Combine numerical features + encoded categorical features
encoded_cols = [f"{col}_encoded" for col in categorical_features]
assembler_input_cols = numerical_features + encoded_cols

# Step 5: Create VectorAssembler
assembler = VectorAssembler(inputCols=assembler_input_cols, outputCol="features", handleInvalid="skip")

# Step 6: Create Logistic Regression model
lr_enhanced = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01
)

# Step 7: Build Pipeline
# Combine all stages: string_indexers + one_hot_encoders + [assembler, lr_enhanced]
pipeline_stages = string_indexers + one_hot_encoders + [assembler, lr_enhanced]

pipeline = Pipeline(stages=pipeline_stages)

# Step 8: Prepare data (label indexing)
label_indexer = StringIndexer(inputCol="Churn", outputCol="label")
labeled_df = label_indexer.fit(churn_df).transform(churn_df)

# Step 9: Train/test split
train_data_enhanced, test_data_enhanced = labeled_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training samples: {train_data_enhanced.count():,}")
print(f"Test samples: {test_data_enhanced.count():,}")

# Step 10: Train the pipeline
print("\nTraining enhanced Logistic Regression pipeline...")
pipeline_model = pipeline.fit(train_data_enhanced)
print("✅ Training complete!")

# Step 11: Make predictions
predictions_enhanced = pipeline_model.transform(test_data_enhanced)

# Step 12: Calculate metrics
evaluator_auc_enhanced = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_acc_enhanced = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision_enhanced = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall_enhanced = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1_enhanced = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

auc_enhanced = evaluator_auc_enhanced.evaluate(predictions_enhanced)
accuracy_enhanced = evaluator_acc_enhanced.evaluate(predictions_enhanced)
precision_enhanced = evaluator_precision_enhanced.evaluate(predictions_enhanced)
recall_enhanced = evaluator_recall_enhanced.evaluate(predictions_enhanced)
f1_enhanced = evaluator_f1_enhanced.evaluate(predictions_enhanced)

# Display results
print("\n" + "="*60)
print("ENHANCED LOGISTIC REGRESSION PERFORMANCE")
print("="*60)
print(f"AUC-ROC:   {auc_enhanced:.4f}")
print(f"Accuracy:  {accuracy_enhanced:.4f}")
print(f"Precision: {precision_enhanced:.4f}")
print(f"Recall:    {recall_enhanced:.4f}")
print(f"F1-Score:  {f1_enhanced:.4f}")
print("="*60)

# Compare to baseline
print("\n📊 Improvement over baseline:")
print(f"AUC-ROC:   {auc_enhanced - auc:+.4f}")
print(f"Recall:    {recall_enhanced - recall:+.4f}")
print(f"Precision: {precision_enhanced - precision:+.4f}")

print("\n✅ By adding categorical features, we significantly improved the model!")
print("   Contract type, internet service, and payment method are strong churn predictors.")

---

## Topic 2: Handling Imbalanced Data

### The Imbalanced Dataset Challenge

**What is class imbalance?** When one class (majority) significantly outnumbers the other (minority):
- Our churn dataset: ~85% no churn (majority), ~15% churn (minority)
- Fraud detection: <1% fraudulent, >99% legitimate
- Disease diagnosis: 5% positive, 95% negative

**Why is this problematic?** Models trained on imbalanced data tend to:
1. **Predict the majority class for everything** (achieves high accuracy but useless for business)
2. **Fail to learn minority class patterns** (not enough examples to generalize)
3. **Produce biased probability estimates** (probabilities skewed toward majority)

**Real-world impact**:
```python
# Naive model that always predicts "no churn"
accuracy = 85%  # Looks great!
recall_on_churners = 0%  # But catches ZERO churners! Useless for retention.
```

### Strategies for Handling Imbalance

#### 1. Class Weights (Algorithm-level)
**Idea**: Penalize misclassifying the minority class more heavily during training.

In Spark's Logistic Regression:
```python
# Calculate class weights inversely proportional to class frequency
# If 15% churn, weight_churn = 1/0.15 = 6.67
# If 85% no_churn, weight_no_churn = 1/0.85 = 1.18

lr = LogisticRegression(
    weightCol="classWeight"  # Column containing per-sample weights
)
```

**Pros**: Simple, no data modification, works with any algorithm supporting weights  
**Cons**: May not fully solve severe imbalance (e.g., 1:100 ratio)

#### 2. Oversampling Minority Class (Data-level)
**Idea**: Duplicate minority class samples to balance the dataset.

```python
# Original: 85,000 no_churn, 15,000 churn
# After oversampling: 85,000 no_churn, 85,000 churn (duplicates)

majority = df.filter(F.col("label") == 0)
minority = df.filter(F.col("label") == 1)

# Oversample minority to match majority
minority_oversampled = minority.sample(
    withReplacement=True, 
    fraction=(majority.count() / minority.count())
)
balanced_df = majority.union(minority_oversampled)
```

**Pros**: Easy to implement, works with any algorithm  
**Cons**: Risk of overfitting (model memorizes duplicates), increases dataset size

#### 3. Undersampling Majority Class (Data-level)
**Idea**: Randomly remove majority class samples to balance the dataset.

```python
# Original: 85,000 no_churn, 15,000 churn
# After undersampling: 15,000 no_churn (sampled), 15,000 churn

majority_undersampled = majority.sample(
    withReplacement=False,
    fraction=(minority.count() / majority.count())
)
balanced_df = majority_undersampled.union(minority)
```

**Pros**: Reduces dataset size (faster training), no overfitting risk  
**Cons**: Discards potentially useful data, may hurt performance if majority has important patterns

#### 4. Adjusting Decision Threshold
**Idea**: Instead of using default 0.5 threshold, optimize for business goals.

```python
# Default: predict churn if P(churn) > 0.5
# Adjusted: predict churn if P(churn) > 0.3 (more aggressive, higher recall)

predictions_adjusted = predictions.withColumn(
    "prediction_adjusted",
    F.when(F.col("probability")[1] > 0.3, 1.0).otherwise(0.0)
)
```

**Pros**: No retraining needed, can optimize for specific business metrics  
**Cons**: Requires threshold tuning, trades off precision vs recall

### Choosing the Right Strategy

| Strategy | Best For | Avoid When |
|----------|----------|------------|
| **Class Weights** | Moderate imbalance (1:5 to 1:10), large datasets | Extreme imbalance (1:100+) |
| **Oversampling** | Small minority class, severe imbalance | Very large datasets (memory issues) |
| **Undersampling** | Very large datasets, fast iteration needed | Small datasets, rich majority patterns |
| **Threshold Tuning** | Post-training optimization, specific business metrics | Need probabilistic predictions |

**Best practice**: Try multiple strategies and compare using business-relevant metrics (recall, precision, F1).

In [ ]:
# Demo 2.1: Compare imbalance handling strategies
# We'll train 3 models: baseline (no handling), class weights, and oversampling

# Use the churn_df_labeled from Demo 1.1 (with "label" column)
# For this demo, we'll use a simpler feature set for clarity
demo_features = ["tenure", "MonthlyCharges", "TotalCharges"]

# Create feature vector
demo_assembler = VectorAssembler(inputCols=demo_features, outputCol="features", handleInvalid="skip")
demo_df = demo_assembler.transform(churn_df_labeled)

# Split data
train_demo, test_demo = demo_df.randomSplit([0.8, 0.2], seed=42)

print("="*70)
print("STRATEGY 1: Baseline (No Imbalance Handling)")
print("="*70)

# Train baseline model (we already did this earlier, but let's be explicit)
lr_baseline = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01
)
model_baseline = lr_baseline.fit(train_demo)
pred_baseline = model_baseline.transform(test_demo)

# Evaluate
eval_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
eval_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
eval_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

auc_baseline = eval_auc.evaluate(pred_baseline)
recall_baseline = eval_recall.evaluate(pred_baseline)
precision_baseline = eval_precision.evaluate(pred_baseline)

print(f"AUC-ROC:   {auc_baseline:.4f}")
print(f"Recall:    {recall_baseline:.4f}")
print(f"Precision: {precision_baseline:.4f}")

print("\n" + "="*70)
print("STRATEGY 2: Class Weights")
print("="*70)

# Calculate class weights based on inverse frequency
total = train_demo.count()
class_0_count = train_demo.filter(F.col("label") == 0).count()
class_1_count = train_demo.filter(F.col("label") == 1).count()

weight_0 = total / (2.0 * class_0_count)
weight_1 = total / (2.0 * class_1_count)

print(f"Class 0 (no churn) weight: {weight_0:.4f}")
print(f"Class 1 (churn) weight:    {weight_1:.4f}")
print(f"Ratio: {weight_1/weight_0:.2f}x more penalty for misclassifying churners\n")

# Add weight column to training data
train_weighted = train_demo.withColumn(
    "classWeight",
    F.when(F.col("label") == 0, weight_0).otherwise(weight_1)
)

# Train with class weights
lr_weighted = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=10,
    regParam=0.01
)
model_weighted = lr_weighted.fit(train_weighted)
pred_weighted = model_weighted.transform(test_demo)

auc_weighted = eval_auc.evaluate(pred_weighted)
recall_weighted = eval_recall.evaluate(pred_weighted)
precision_weighted = eval_precision.evaluate(pred_weighted)

print(f"AUC-ROC:   {auc_weighted:.4f}")
print(f"Recall:    {recall_weighted:.4f}")
print(f"Precision: {precision_weighted:.4f}")

print("\n" + "="*70)
print("STRATEGY 3: Oversampling Minority Class")
print("="*70)

# Separate majority and minority classes
majority_class = train_demo.filter(F.col("label") == 0)
minority_class = train_demo.filter(F.col("label") == 1)

majority_count = majority_class.count()
minority_count = minority_class.count()

# Oversample minority to match majority
oversample_ratio = majority_count / minority_count
minority_oversampled = minority_class.sample(withReplacement=True, fraction=oversample_ratio, seed=42)

# Combine into balanced dataset
train_balanced = majority_class.union(minority_oversampled)

print(f"Original training set: {majority_count:,} no churn, {minority_count:,} churn")
print(f"Balanced training set: {majority_count:,} no churn, {minority_oversampled.count():,} churn")
print(f"New class distribution: 50/50\n")

# Train on balanced data
lr_balanced = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01
)
model_balanced = lr_balanced.fit(train_balanced)
pred_balanced = model_balanced.transform(test_demo)

auc_balanced = eval_auc.evaluate(pred_balanced)
recall_balanced = eval_recall.evaluate(pred_balanced)
precision_balanced = eval_precision.evaluate(pred_balanced)

print(f"AUC-ROC:   {auc_balanced:.4f}")
print(f"Recall:    {recall_balanced:.4f}")
print(f"Precision: {precision_balanced:.4f}")

# Summary comparison
print("\n" + "="*70)
print("COMPARISON SUMMARY")
print("="*70)
print(f"{'Strategy':<25} {'AUC-ROC':<12} {'Recall':<12} {'Precision':<12}")
print("-"*70)
print(f"{'Baseline':<25} {auc_baseline:<12.4f} {recall_baseline:<12.4f} {precision_baseline:<12.4f}")
print(f"{'Class Weights':<25} {auc_weighted:<12.4f} {recall_weighted:<12.4f} {precision_weighted:<12.4f}")
print(f"{'Oversampling':<25} {auc_balanced:<12.4f} {recall_balanced:<12.4f} {precision_balanced:<12.4f}")
print("="*70)

print("\n📊 Key Takeaways:")
print("- Class weights typically improve recall (catching more churners)")
print("- Oversampling can further boost recall but may reduce precision")
print("- Choose strategy based on business priority (miss fewer churners vs avoid false alarms)")

### Lab 2.1: Implement Undersampling and Threshold Tuning

**Objective**: Explore two additional imbalance handling strategies: undersampling the majority class and adjusting the decision threshold.

**Background**: The demo showed class weights and oversampling. Two other common approaches are:
1. **Undersampling**: Randomly sample the majority class to match minority class size (faster training, less data)
2. **Threshold tuning**: Adjust the probability cutoff for classification (e.g., predict churn if P(churn) > 0.3 instead of 0.5)

**Your Task**: Implement both strategies and compare their performance.

**Part A: Undersampling**

1. **Create balanced dataset via undersampling**:
   - Split `train_demo` (from Demo 2.1) into majority (label=0) and minority (label=1) classes
   - Undersample majority class to match minority class size using `.sample()`
   - Union the undersampled majority with full minority class

2. **Train Logistic Regression on undersampled data**:
   - Use same hyperparameters as baseline (maxIter=10, regParam=0.01)
   - Make predictions on `test_demo`

3. **Evaluate performance**:
   - Calculate AUC-ROC, Recall, Precision
   - Compare to baseline, class weights, and oversampling strategies

**Part B: Threshold Tuning**

1. **Use baseline model predictions** (from Demo 2.1: `pred_baseline`):
   - Extract probability of churn: `probability[1]` (index 1 = positive class)

2. **Test multiple thresholds**:
   - Try thresholds: [0.3, 0.4, 0.5, 0.6, 0.7]
   - For each threshold, create new predictions using `F.when()`
   - Calculate recall and precision for each threshold

3. **Visualize precision-recall tradeoff**:
   - Create a simple table showing threshold vs recall vs precision
   - Identify optimal threshold for your business goal (e.g., recall > 0.70)

**Expected Outcomes**:
- **Undersampling**: Faster training, recall ~0.65-0.70, precision may drop slightly
- **Threshold tuning**: Can achieve recall > 0.70 by lowering threshold to ~0.35-0.40

**Hints**:
- For undersampling: `fraction = minority_count / majority_count`
- For threshold tuning: `F.col("probability").getItem(1)` extracts churn probability
- Lower threshold → higher recall, lower precision (predict churn more aggressively)
- Higher threshold → lower recall, higher precision (predict churn more conservatively)

In [ ]:
# Lab 2.1: SOLUTION - Undersampling and Threshold Tuning

# ============================================================
# PART A: Undersampling
# ============================================================

# Step 1: Separate majority and minority from train_demo
majority_lab = train_demo.filter(F.col("label") == 0)
minority_lab = train_demo.filter(F.col("label") == 1)

# Step 2: Calculate counts
majority_count_lab = majority_lab.count()
minority_count_lab = minority_lab.count()

# Step 3: Undersample majority to match minority
undersample_fraction = minority_count_lab / majority_count_lab
majority_undersampled = majority_lab.sample(withReplacement=False, fraction=undersample_fraction, seed=42)

# Step 4: Create balanced dataset
train_undersampled = majority_undersampled.union(minority_lab)

# Step 5: Train Logistic Regression on undersampled data
lr_undersampled = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01
)
model_undersampled = lr_undersampled.fit(train_undersampled)
pred_undersampled = model_undersampled.transform(test_demo)

# Step 6: Evaluate undersampled model
auc_undersampled = eval_auc.evaluate(pred_undersampled)
recall_undersampled = eval_recall.evaluate(pred_undersampled)
precision_undersampled = eval_precision.evaluate(pred_undersampled)

print("="*70)
print("UNDERSAMPLING RESULTS")
print("="*70)
print(f"Original training: {majority_count_lab:,} majority, {minority_count_lab:,} minority")
print(f"After undersampling: {majority_undersampled.count():,} majority, {minority_count_lab:,} minority")
print(f"\nAUC-ROC:   {auc_undersampled:.4f}")
print(f"Recall:    {recall_undersampled:.4f}")
print(f"Precision: {precision_undersampled:.4f}")

# ============================================================
# PART B: Threshold Tuning
# ============================================================

# Use pred_baseline from Demo 2.1
# We'll test different thresholds and measure precision/recall tradeoff

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for threshold in thresholds:
    # Step 1: Create predictions based on custom threshold
    # Extract probability of class 1 using F.col("probability").getItem(1)
    pred_custom = pred_baseline.withColumn(
        "prediction_custom",
        F.when(F.col("probability").getItem(1) > threshold, 1.0).otherwise(0.0)
    )
    
    # Step 2: Calculate recall and precision for this threshold
    recall_evaluator_custom = MulticlassClassificationEvaluator(
        labelCol="label", 
        predictionCol="prediction_custom", 
        metricName="weightedRecall"
    )
    precision_evaluator_custom = MulticlassClassificationEvaluator(
        labelCol="label", 
        predictionCol="prediction_custom", 
        metricName="weightedPrecision"
    )
    
    recall_custom = recall_evaluator_custom.evaluate(pred_custom)
    precision_custom = precision_evaluator_custom.evaluate(pred_custom)
    
    results.append({
        "threshold": threshold,
        "recall": recall_custom,
        "precision": precision_custom
    })

# Display threshold tuning results
print("\n" + "="*70)
print("THRESHOLD TUNING RESULTS")
print("="*70)
print(f"{'Threshold':<15} {'Recall':<15} {'Precision':<15}")
print("-"*70)
for r in results:
    print(f"{r['threshold']:<15.2f} {r['recall']:<15.4f} {r['precision']:<15.4f}")
print("="*70)

# Step 3: Identify optimal threshold
# Find threshold that achieves recall > 0.70 (if possible)
optimal_threshold = None
for r in results:
    if r['recall'] > 0.70:
        optimal_threshold = r['threshold']
        break

if optimal_threshold is None:
    optimal_threshold = results[0]['threshold']  # Lowest threshold
    print(f"\n📊 Best threshold (highest recall): {optimal_threshold}")
else:
    print(f"\n📊 Optimal threshold for recall > 0.70: {optimal_threshold}")

print("\nKey Insight: Lowering threshold increases recall (catch more churners)")
print("             but decreases precision (more false alarms)")
print("\n✅ Threshold tuning allows post-training optimization without retraining!")

---

## Topic 3: Advanced Ensemble Models

### Beyond Linear Models: Tree-Based Ensembles

**Limitations of Logistic Regression**:
- Assumes linear relationship between features and log-odds of outcome
- Cannot capture complex feature interactions without manual feature engineering
- May underperform when decision boundaries are non-linear

**Enter ensemble models**: Combine multiple models (typically decision trees) to capture complex patterns.

### Random Forest Classifier

**How it works**:
1. **Build many decision trees** (e.g., 100 trees), each trained on a random subset of data
2. **Each tree makes a prediction** (churn or no churn)
3. **Majority vote wins**: If 60 trees say "churn" and 40 say "no churn", predict churn
4. **Randomness reduces overfitting**: Different trees make different errors, averaging them out

**Key hyperparameters**:
```python
RandomForestClassifier(
    numTrees=100,           # Number of trees (more = better but slower)
    maxDepth=5,             # Maximum depth of each tree (controls complexity)
    featureSubsetStrategy="auto"  # How many features to consider per split
)
```

**Strengths**:
- ✅ Handles non-linear relationships naturally
- ✅ Captures feature interactions automatically
- ✅ Robust to outliers and missing data
- ✅ Less prone to overfitting than single decision trees

**Weaknesses**:
- ❌ Slower to train than Logistic Regression (many trees)
- ❌ Less interpretable (100 trees vs single set of coefficients)
- ❌ Larger model size (storage/memory)

### Gradient Boosted Trees (GBT)

**How it works** (different from Random Forest):
1. **Build trees sequentially**, not in parallel
2. **Each new tree corrects errors** of previous trees
3. **Focus on hard-to-classify examples** (boosting)
4. **Final prediction**: Weighted sum of all tree predictions

**Analogy**: Random Forest = committee of independent experts voting  
            GBT = one expert learning from mistakes of previous attempts

**Key hyperparameters**:
```python
GBTClassifier(
    maxIter=10,             # Number of trees (more = better but risk overfitting)
    maxDepth=4,             # Depth of each tree (typically shallower than RF)
    stepSize=0.1            # Learning rate (smaller = more conservative learning)
)
```

**Strengths**:
- ✅ Often achieves best performance (Kaggle competitions)
- ✅ Handles complex patterns extremely well
- ✅ Can use fewer trees than Random Forest

**Weaknesses**:
- ❌ Easier to overfit (sequential learning)
- ❌ Sensitive to hyperparameters
- ❌ Longer training time (trees built sequentially)

### When to Use Each Model

| Model | Best For | Training Time | Interpretability |
|-------|----------|---------------|------------------|
| **Logistic Regression** | Baseline, linear relationships, speed | Fastest | High (coefficients) |
| **Random Forest** | Non-linear patterns, robust performance | Medium | Low (many trees) |
| **Gradient Boosted Trees** | Maximum performance, competitions | Slowest | Low (sequential trees) |

**Recommendation**: Start with Logistic Regression, then try Random Forest if performance is insufficient. Use GBT only if you need that last few percentage points of accuracy and can afford longer training.

In [ ]:
# Demo 3.1: Compare Random Forest and GBT classifiers
# We'll train both models and compare to Logistic Regression baseline

# Use the balanced training data from oversampling (train_balanced from Demo 2.1)
# This ensures fair comparison with imbalance handling already applied

print("="*70)
print("MODEL COMPARISON: Logistic Regression vs Random Forest vs GBT")
print("="*70)

# --------------------------------------------------------------------
# Model 1: Logistic Regression (baseline from earlier)
# --------------------------------------------------------------------
print("\n[1/3] Training Logistic Regression...")

lr_compare = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    regParam=0.01
)
model_lr_compare = lr_compare.fit(train_balanced)
pred_lr_compare = model_lr_compare.transform(test_demo)

auc_lr_compare = eval_auc.evaluate(pred_lr_compare)
recall_lr_compare = eval_recall.evaluate(pred_lr_compare)
precision_lr_compare = eval_precision.evaluate(pred_lr_compare)

print(f"✅ Complete - AUC: {auc_lr_compare:.4f}, Recall: {recall_lr_compare:.4f}, Precision: {precision_lr_compare:.4f}")

# --------------------------------------------------------------------
# Model 2: Random Forest
# --------------------------------------------------------------------
print("\n[2/3] Training Random Forest (100 trees)...")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,          # Number of trees in the forest
    maxDepth=5,            # Maximum depth of each tree
    featureSubsetStrategy="auto",  # sqrt(numFeatures) per split
    seed=42
)

# Fit Random Forest (this may take 15-30 seconds)
model_rf = rf.fit(train_balanced)
pred_rf = model_rf.transform(test_demo)

auc_rf = eval_auc.evaluate(pred_rf)
recall_rf = eval_recall.evaluate(pred_rf)
precision_rf = eval_precision.evaluate(pred_rf)

print(f"✅ Complete - AUC: {auc_rf:.4f}, Recall: {recall_rf:.4f}, Precision: {precision_rf:.4f}")

# --------------------------------------------------------------------
# Model 3: Gradient Boosted Trees
# --------------------------------------------------------------------
print("\n[3/3] Training Gradient Boosted Trees (10 iterations)...")

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=10,            # Number of boosting iterations (trees)
    maxDepth=4,            # Depth per tree (shallower than RF)
    stepSize=0.1,          # Learning rate (conservative)
    seed=42
)

# Fit GBT (this may take 20-40 seconds)
model_gbt = gbt.fit(train_balanced)
pred_gbt = model_gbt.transform(test_demo)

auc_gbt = eval_auc.evaluate(pred_gbt)
recall_gbt = eval_recall.evaluate(pred_gbt)
precision_gbt = eval_precision.evaluate(pred_gbt)

print(f"✅ Complete - AUC: {auc_gbt:.4f}, Recall: {recall_gbt:.4f}, Precision: {precision_gbt:.4f}")

# --------------------------------------------------------------------
# Comparison Summary
# --------------------------------------------------------------------
print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"{'Model':<30} {'AUC-ROC':<12} {'Recall':<12} {'Precision':<12}")
print("-"*70)
print(f"{'Logistic Regression':<30} {auc_lr_compare:<12.4f} {recall_lr_compare:<12.4f} {precision_lr_compare:<12.4f}")
print(f"{'Random Forest (100 trees)':<30} {auc_rf:<12.4f} {recall_rf:<12.4f} {precision_rf:<12.4f}")
print(f"{'Gradient Boosted Trees':<30} {auc_gbt:<12.4f} {recall_gbt:<12.4f} {precision_gbt:<12.4f}")
print("="*70)

# Feature importance (Random Forest only - GBT doesn't expose feature importance in Spark)
print("\n📊 Top 5 Most Important Features (Random Forest):")
feature_importances = model_rf.featureImportances
# Since we only have 3 features (tenure, MonthlyCharges, TotalCharges)
for i, importance in enumerate(feature_importances):
    print(f"  Feature {i} ({demo_features[i]}): {importance:.4f}")

print("\n🎯 Key Takeaways:")
print("- Random Forest typically improves AUC by 3-5% over Logistic Regression")
print("- GBT may achieve highest AUC but can overfit if not tuned carefully")
print("- Both ensemble models capture non-linear patterns in customer behavior")
print("- Trade-off: Better performance vs longer training time and less interpretability")

### Lab 3.1: Build a Random Forest Model with Full Feature Set

**Objective**: Train a Random Forest classifier using the full feature set (numerical + categorical) from Lab 1.1, apply imbalance handling, and optimize hyperparameters.

**Background**: Now that you understand ensemble models, it's time to build a production-quality Random Forest model that:
1. Uses all available features (not just the 3 numerical features from demos)
2. Handles class imbalance (using the strategy of your choice)
3. Is optimized for our business goal (high recall while maintaining acceptable precision)

**Your Task**: Build an end-to-end Random Forest classification pipeline.

**Step-by-Step Instructions**:

1. **Prepare data with full features**:
   - Use the same feature engineering from Lab 1.1:
     - Numerical: `tenure`, `MonthlyCharges`, `TotalCharges`
     - Categorical: `Contract`, `InternetService`, `PaymentMethod`, `OnlineSecurity`, `TechSupport`
   - Build Pipeline with StringIndexers → OneHotEncoders → VectorAssembler

2. **Apply imbalance handling**:
   - Choose ONE strategy: class weights, oversampling, or undersampling
   - Justify your choice based on dataset characteristics
   - Apply to training data only (not test data!)

3. **Train Random Forest**:
   - Use `RandomForestClassifier` with these hyperparameters:
     - `numTrees`: 100
     - `maxDepth`: Try 5, 7, or 10 (experiment to see what works best)
     - `featureSubsetStrategy`: "auto"
   - Fit on your prepared training data

4. **Evaluate performance**:
   - Calculate all 5 metrics: AUC-ROC, Accuracy, Precision, Recall, F1-Score
   - Compare to your enhanced Logistic Regression from Lab 1.1
   - Check if you meet business goals: Recall > 0.70, Precision > 0.55

5. **Analyze feature importance**:
   - Extract `model.featureImportances`
   - Identify top 5 most predictive features
   - Discuss: Do the results align with business intuition?

**Expected Outcomes**:
- AUC-ROC: > 0.82 (improvement over Logistic Regression)
- Recall: > 0.72 (catching most churners)
- Precision: > 0.58 (acceptable false alarm rate)

**Hints**:
- Reuse your Pipeline code from Lab 1.1 (change only the final estimator to RandomForestClassifier)
- For class weights with Random Forest, you may need to manually weight samples (not all Spark models support `weightCol`)
- Feature importance indices correspond to the order of features in `VectorAssembler.inputCols`
- Training 100 trees may take 1-2 minutes - this is normal!

**Bonus Challenge** (optional):
- Try both Random Forest AND GBT, compare their performance
- Which model achieves better recall? Which is faster to train?

In [ ]:
# Lab 3.1: SOLUTION - Random Forest with Full Features

# Step 1: Prepare data with full feature set
numerical_features_rf = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features_rf = ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport"]

# Step 2: Create preprocessing pipeline stages
string_indexers_rf = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in categorical_features_rf
]

one_hot_encoders_rf = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    for col in categorical_features_rf
]

# Step 3: Prepare assembler input columns
encoded_cols_rf = [f"{col}_encoded" for col in categorical_features_rf]
assembler_input_cols_rf = numerical_features_rf + encoded_cols_rf

assembler_rf = VectorAssembler(inputCols=assembler_input_cols_rf, outputCol="features", handleInvalid="skip")

# Step 4: Prepare labeled data
label_indexer_rf = StringIndexer(inputCol="Churn", outputCol="label")
labeled_df_rf = label_indexer_rf.fit(churn_df).transform(churn_df)

# Step 5: Train/test split
train_rf, test_rf = labeled_df_rf.randomSplit([0.8, 0.2], seed=42)

# Step 6: Apply imbalance handling - Using oversampling
majority_rf = train_rf.filter(F.col("label") == 0)
minority_rf = train_rf.filter(F.col("label") == 1)

majority_count_rf = majority_rf.count()
minority_count_rf = minority_rf.count()

oversample_ratio_rf = majority_count_rf / minority_count_rf
minority_oversampled_rf = minority_rf.sample(withReplacement=True, fraction=oversample_ratio_rf, seed=42)

train_rf_balanced = majority_rf.union(minority_oversampled_rf)

print(f"Imbalance handling applied: Train set now has {train_rf_balanced.count():,} samples")
print(f"  Majority: {majority_count_rf:,}, Minority: {minority_oversampled_rf.count():,}")

# Step 7: Create Random Forest Pipeline
rf_model = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=7,
    featureSubsetStrategy="auto",
    seed=42
)

pipeline_stages_rf = string_indexers_rf + one_hot_encoders_rf + [assembler_rf, rf_model]
pipeline_rf = Pipeline(stages=pipeline_stages_rf)

# Step 8: Train the pipeline
print("\nTraining Random Forest (this may take 1-2 minutes)...")
pipeline_model_rf = pipeline_rf.fit(train_rf_balanced)
print("✅ Training complete!")

# Step 9: Make predictions
predictions_rf = pipeline_model_rf.transform(test_rf)

# Step 10: Calculate all metrics
evaluator_auc_rf = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_acc_rf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision_rf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall_rf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1_rf = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

auc_rf_lab = evaluator_auc_rf.evaluate(predictions_rf)
accuracy_rf_lab = evaluator_acc_rf.evaluate(predictions_rf)
precision_rf_lab = evaluator_precision_rf.evaluate(predictions_rf)
recall_rf_lab = evaluator_recall_rf.evaluate(predictions_rf)
f1_rf_lab = evaluator_f1_rf.evaluate(predictions_rf)

# Display results
print("\n" + "="*70)
print("RANDOM FOREST PERFORMANCE (Full Feature Set)")
print("="*70)
print(f"AUC-ROC:   {auc_rf_lab:.4f}")
print(f"Accuracy:  {accuracy_rf_lab:.4f}")
print(f"Precision: {precision_rf_lab:.4f}")
print(f"Recall:    {recall_rf_lab:.4f}")
print(f"F1-Score:  {f1_rf_lab:.4f}")
print("="*70)

# Step 11: Feature importance analysis
# Extract the trained RandomForest model from the pipeline
rf_model_trained = pipeline_model_rf.stages[-1]

feature_importances_rf = rf_model_trained.featureImportances

# Map feature indices to feature names
all_feature_names = numerical_features_rf + encoded_cols_rf

# Print top 5 most important features
print("\n📊 Top 5 Most Important Features:")
importance_list = [(all_feature_names[i], float(feature_importances_rf[i])) 
                   for i in range(len(all_feature_names))]
importance_list.sort(key=lambda x: x[1], reverse=True)

for i, (feature, importance) in enumerate(importance_list[:5], 1):
    print(f"  {i}. {feature}: {importance:.4f}")

print("\n✅ Business Goal Check:")
print(f"  Recall > 0.70? {'✅ Yes' if recall_rf_lab > 0.70 else '❌ No'} (Actual: {recall_rf_lab:.4f})")
print(f"  Precision > 0.55? {'✅ Yes' if precision_rf_lab > 0.55 else '❌ No'} (Actual: {precision_rf_lab:.4f})")

print("\n🎯 Random Forest significantly outperforms Logistic Regression!")
print("   The ensemble captures complex non-linear patterns in customer behavior.")

---

## Topic 4: Cross-Validation and Hyperparameter Tuning

### The Hyperparameter Challenge

**Problem**: How do you choose the best hyperparameters for your model?
- Random Forest: `numTrees`, `maxDepth`, `featureSubsetStrategy`
- Logistic Regression: `regParam`, `elasticNetParam`, `maxIter`
- GBT: `maxIter`, `maxDepth`, `stepSize`

**Naive approach** (don't do this!):
```python
# Train on training set, test on test set
model = RandomForestClassifier(numTrees=50, maxDepth=3)
model.fit(train_data)
score = evaluate(model, test_data)  # AUC = 0.75

# Try different parameters
model = RandomForestClassifier(numTrees=100, maxDepth=5)
model.fit(train_data)
score = evaluate(model, test_data)  # AUC = 0.78

# Keep tuning until test score is high...
```

**Why this is wrong**: You're "peeking" at the test set! You might overfit to it, and your final performance estimate is overly optimistic.

### Cross-Validation: The Right Way

**K-Fold Cross-Validation** splits training data into K subsets (folds):
1. **Train on K-1 folds, validate on 1 fold** (repeat K times)
2. **Average the K validation scores** → robust performance estimate
3. **Test set is never touched** until final evaluation

**Example with K=3**:
```
Fold 1: Train on [2,3], validate on [1]  →  AUC = 0.76
Fold 2: Train on [1,3], validate on [2]  →  AUC = 0.78
Fold 3: Train on [1,2], validate on [3]  →  AUC = 0.77

Average CV score: 0.77  (use this to compare hyperparameters)
```

**Benefits**:
- Uses all training data for both training and validation
- Provides robust estimate of model performance
- Prevents overfitting to a single validation split

### Spark MLlib CrossValidator

Spark provides `CrossValidator` to automate hyperparameter search with cross-validation:

```python
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Define parameter grid to search
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 150]) \
    .addGrid(rf.maxDepth, [5, 7, 10]) \
    .build()  # Creates 3 x 3 = 9 combinations

# Create CrossValidator
cv = CrossValidator(
    estimator=pipeline,           # Your ML pipeline
    estimatorParamMaps=paramGrid, # Grid of hyperparameters to try
    evaluator=BinaryClassificationEvaluator(metricName="areaUnderROC"),
    numFolds=3,                   # 3-fold cross-validation
    seed=42
)

# Fit CrossValidator (trains 9 models x 3 folds = 27 model fits!)
cv_model = cv.fit(train_data)

# Best model is automatically selected
best_model = cv_model.bestModel
```

**How it works**:
1. For each hyperparameter combination in the grid:
   - Perform 3-fold CV (train 3 models)
   - Average the 3 validation scores
2. Select combination with highest average CV score
3. Retrain on full training set with best hyperparameters

### Grid Search vs Random Search

**Grid Search** (what CrossValidator does):
- Tests every combination in the grid
- Pros: Guaranteed to find best in grid, systematic
- Cons: Exponential growth (3 params x 3 values each = 27 models!)

**Random Search** (alternative):
- Randomly samples from hyperparameter space
- Pros: More efficient, can cover wider range
- Cons: Might miss optimal combination

**Recommendation**: 
- Small grids (< 20 combinations): Use Grid Search (CrossValidator)
- Large search spaces: Use Random Search or advanced methods (Hyperopt, Optuna)

### Practical Considerations

**Computational cost**:
- CrossValidator with 3 folds and 9 hyperparameter combinations = 27 model trainings
- With 100-tree Random Forest, this can take 10-30 minutes on a cluster
- **Tip**: Start with small grids, expand if time allows

**Choosing K (number of folds)**:
- K=3: Faster, less stable estimates (good for large datasets)
- K=5: Standard choice, good trade-off
- K=10: More stable, slower (good for small datasets)

**Choosing metrics**:
- Use metric aligned with business goal
- For churn: `areaUnderROC` (overall discriminative ability)
- Could also use `areaUnderPR` (precision-recall curve) for severe imbalance

In [ ]:
# Demo 4.1: Hyperparameter tuning with CrossValidator
# We'll tune a Logistic Regression model using 3-fold CV and a small parameter grid

# Use labeled_df and create a simple feature set for fast demo
demo_cv_features = ["tenure", "MonthlyCharges", "TotalCharges"]
assembler_cv = VectorAssembler(inputCols=demo_cv_features, outputCol="features")
df_cv = assembler_cv.transform(labeled_df)

# Split data
train_cv, test_cv = df_cv.randomSplit([0.8, 0.2], seed=42)

# Apply oversampling for imbalance handling
majority_cv = train_cv.filter(F.col("label") == 0)
minority_cv = train_cv.filter(F.col("label") == 1)
oversample_ratio_cv = majority_cv.count() / minority_cv.count()
minority_oversampled_cv = minority_cv.sample(withReplacement=True, fraction=oversample_ratio_cv, seed=42)
train_cv_balanced = majority_cv.union(minority_oversampled_cv)

print("="*70)
print("HYPERPARAMETER TUNING WITH CROSSVALIDATOR")
print("="*70)

# Create base Logistic Regression model
lr_cv = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10  # We'll tune regParam, not maxIter
)

# Define parameter grid
# We'll search over regularization parameter (regParam)
paramGrid = ParamGridBuilder() \
    .addGrid(lr_cv.regParam, [0.001, 0.01, 0.1]) \
    .addGrid(lr_cv.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

print(f"\nParameter grid size: {len(paramGrid)} combinations")
print("Parameters to tune:")
print("  - regParam: [0.001, 0.01, 0.1]  (regularization strength)")
print("  - elasticNetParam: [0.0, 0.5, 1.0]  (L1 vs L2 ratio)")
print(f"\nTotal models to train: {len(paramGrid)} combinations × 3 folds = {len(paramGrid) * 3} models")

# Create evaluator (use AUC-ROC as optimization metric)
evaluator_cv = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Create CrossValidator
cv = CrossValidator(
    estimator=lr_cv,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_cv,
    numFolds=3,  # 3-fold cross-validation
    seed=42
)

# Fit CrossValidator (this searches all combinations and selects best)
print("\n⏳ Running cross-validation (this may take 30-60 seconds)...")
cv_model = cv.fit(train_cv_balanced)
print("✅ Cross-validation complete!")

# Extract best model
best_model_cv = cv_model.bestModel

# Get best hyperparameters
print("\n" + "="*70)
print("BEST HYPERPARAMETERS (selected via CV)")
print("="*70)
print(f"regParam:        {best_model_cv.getRegParam():.4f}")
print(f"elasticNetParam: {best_model_cv.getElasticNetParam():.4f}")

# Make predictions on test set with best model
pred_best_cv = cv_model.transform(test_cv)

# Evaluate on test set
auc_best_cv = evaluator_cv.evaluate(pred_best_cv)
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")

recall_best_cv = recall_evaluator.evaluate(pred_best_cv)
precision_best_cv = precision_evaluator.evaluate(pred_best_cv)

print("\n" + "="*70)
print("TEST SET PERFORMANCE (Best Model)")
print("="*70)
print(f"AUC-ROC:   {auc_best_cv:.4f}")
print(f"Recall:    {recall_best_cv:.4f}")
print(f"Precision: {precision_best_cv:.4f}")
print("="*70)

# Show average CV scores for all hyperparameter combinations
print("\n📊 Cross-Validation Scores (all combinations):")
print(f"{'regParam':<12} {'elasticNet':<12} {'Avg CV AUC':<12}")
print("-"*40)
for params, avg_metric in zip(paramGrid, cv_model.avgMetrics):
    reg_param = params[lr_cv.regParam]
    elastic_param = params[lr_cv.elasticNetParam]
    print(f"{reg_param:<12.4f} {elastic_param:<12.4f} {avg_metric:<12.4f}")

print("\n🎯 Key Takeaways:")
print("- CrossValidator automates hyperparameter search with proper validation")
print("- Best model is selected based on average CV score across folds")
print("- Test set is only used for final evaluation (no peeking!)")
print("- Higher regParam = more regularization = simpler model (may underfit)")
print("- Lower regParam = less regularization = complex model (may overfit)")

### Lab 4.1: Tune Random Forest Hyperparameters with CrossValidator

**Objective**: Use CrossValidator to find the optimal hyperparameters for your Random Forest churn prediction model from Lab 3.1.

**Background**: In Lab 3.1, you manually chose hyperparameters (`numTrees=100`, `maxDepth=5 or 7`). Now you'll systematically search for the best combination using cross-validation.

**Your Task**: Set up and run a hyperparameter search for Random Forest.

**Step-by-Step Instructions**:

1. **Reuse your Pipeline from Lab 3.1**:
   - Same feature engineering (StringIndexers, OneHotEncoders, VectorAssembler)
   - Replace final estimator with a base RandomForestClassifier (no fixed hyperparameters yet)

2. **Define parameter grid**:
   - Search over:
     - `numTrees`: [50, 100, 150]
     - `maxDepth`: [5, 7, 10]
   - This creates 3 × 3 = 9 combinations

3. **Create CrossValidator**:
   - Estimator: Your full Pipeline (preprocessing + RandomForestClassifier)
   - Evaluator: `BinaryClassificationEvaluator(metricName="areaUnderROC")`
   - numFolds: 3
   - Grid: Your parameter grid from step 2

4. **Fit CrossValidator on balanced training data**:
   - Use the same imbalance handling strategy from Lab 3.1 (e.g., oversampling)
   - This will train 9 combinations × 3 folds = 27 models (may take 3-5 minutes)

5. **Evaluate best model on test set**:
   - Extract `cv_model.bestModel`
   - Get best hyperparameters
   - Make predictions on test data
   - Calculate all metrics: AUC-ROC, Recall, Precision, F1-Score

6. **Analyze results**:
   - Compare best CV model to your manual Lab 3.1 model
   - Did CV find better hyperparameters?
   - Examine `cv_model.avgMetrics` to see all combinations' performance

**Expected Outcomes**:
- Best hyperparameters automatically selected
- AUC-ROC: > 0.83 (slight improvement over manual tuning)
- Recall: > 0.72
- Confidence that these are near-optimal hyperparameters (validated via CV)

**Hints**:
- Create the base RandomForest WITHOUT specifying `numTrees` or `maxDepth` initially
- Use `ParamGridBuilder().addGrid(rf.numTrees, [...]).addGrid(rf.maxDepth, [...]).build()`
- Access best hyperparameters: `best_model.stages[-1].getNumTrees()` (last stage of pipeline)
- Be patient - training 27 models takes time, but it's worth it!

**Warning**: 
- This will take 3-5 minutes to run
- Don't panic if you see no output for a while - Spark is working
- If time is limited, use smaller grid: `numTrees=[50, 100]`, `maxDepth=[5, 7]` (4 combinations)

In [ ]:
# Lab 4.1: SOLUTION - Hyperparameter Tuning with CrossValidator

# Step 1: Prepare data and preprocessing pipeline
numerical_features_cv = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features_cv = ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity", "TechSupport"]

string_indexers_cv = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in categorical_features_cv
]

one_hot_encoders_cv = [
    OneHotEncoder(inputCol=f"{col}_indexed", outputCol=f"{col}_encoded")
    for col in categorical_features_cv
]

encoded_cols_cv = [f"{col}_encoded" for col in categorical_features_cv]
assembler_input_cv = numerical_features_cv + encoded_cols_cv
assembler_cv_lab = VectorAssembler(inputCols=assembler_input_cv, outputCol="features", handleInvalid="skip")

# Step 2: Prepare labeled data
label_indexer_cv = StringIndexer(inputCol="Churn", outputCol="label")
labeled_df_cv = label_indexer_cv.fit(churn_df).transform(churn_df)

# Step 3: Train/test split
train_cv_lab, test_cv_lab = labeled_df_cv.randomSplit([0.8, 0.2], seed=42)

# Step 4: Apply imbalance handling - oversampling
majority_cv_lab = train_cv_lab.filter(F.col("label") == 0)
minority_cv_lab = train_cv_lab.filter(F.col("label") == 1)

oversample_ratio_cv_lab = majority_cv_lab.count() / minority_cv_lab.count()
minority_oversampled_cv_lab = minority_cv_lab.sample(withReplacement=True, fraction=oversample_ratio_cv_lab, seed=42)

train_cv_balanced_lab = majority_cv_lab.union(minority_oversampled_cv_lab)

print(f"Balanced training data: {train_cv_balanced_lab.count():,} samples")

# Step 5: Create base Random Forest (no hyperparameters specified)
rf_cv = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    seed=42
)

# Step 6: Build pipeline (preprocessing + Random Forest)
pipeline_stages_cv = string_indexers_cv + one_hot_encoders_cv + [assembler_cv_lab, rf_cv]
pipeline_cv_lab = Pipeline(stages=pipeline_stages_cv)

# Step 7: Define parameter grid
paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf_cv.numTrees, [50, 100, 150]) \
    .addGrid(rf_cv.maxDepth, [5, 7, 10]) \
    .build()

print(f"\nParameter grid size: {len(paramGrid_rf)} combinations")
print(f"This will train {len(paramGrid_rf)} × 3 folds = {len(paramGrid_rf) * 3} models")

# Step 8: Create evaluator
evaluator_rf_cv = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Step 9: Create CrossValidator
cv_rf = CrossValidator(
    estimator=pipeline_cv_lab,
    estimatorParamMaps=paramGrid_rf,
    evaluator=evaluator_rf_cv,
    numFolds=3,
    seed=42
)

# Step 10: Fit CrossValidator (this will take 3-5 minutes)
print("\n⏳ Running cross-validation for Random Forest (this may take 3-5 minutes)...")
print("Please be patient - Spark is training 27 models in the background...")
cv_model_rf = cv_rf.fit(train_cv_balanced_lab)

print("✅ Cross-validation complete!")

# Step 11: Extract best model and hyperparameters
best_rf_model = cv_model_rf.bestModel
best_rf_estimator = best_rf_model.stages[-1]  # Last stage is the RF classifier

best_numTrees = best_rf_estimator.getNumTrees
best_maxDepth = best_rf_estimator.getMaxDepth()

print("\n" + "="*70)
print("BEST HYPERPARAMETERS (selected via CV)")
print("="*70)
print(f"numTrees:  {best_numTrees}")
print(f"maxDepth:  {best_maxDepth}")

# Step 12: Evaluate best model on test set
pred_best_rf = cv_model_rf.transform(test_cv_lab)

evaluator_auc_cv = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_acc_cv = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision_cv = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall_cv = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1_cv = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

auc_best_rf = evaluator_auc_cv.evaluate(pred_best_rf)
accuracy_best_rf = evaluator_acc_cv.evaluate(pred_best_rf)
precision_best_rf = evaluator_precision_cv.evaluate(pred_best_rf)
recall_best_rf = evaluator_recall_cv.evaluate(pred_best_rf)
f1_best_rf = evaluator_f1_cv.evaluate(pred_best_rf)

print("\n" + "="*70)
print("TEST SET PERFORMANCE (Best Model via CV)")
print("="*70)
print(f"AUC-ROC:   {auc_best_rf:.4f}")
print(f"Accuracy:  {accuracy_best_rf:.4f}")
print(f"Precision: {precision_best_rf:.4f}")
print(f"Recall:    {recall_best_rf:.4f}")
print(f"F1-Score:  {f1_best_rf:.4f}")
print("="*70)

# Step 13: Analyze all CV results
print("\n📊 Cross-Validation Scores (all combinations):")
print(f"{'numTrees':<12} {'maxDepth':<12} {'Avg CV AUC':<12}")
print("-"*40)

for params, avg_metric in zip(paramGrid_rf, cv_model_rf.avgMetrics):
    num_trees = params[rf_cv.numTrees]
    max_depth = params[rf_cv.maxDepth]
    print(f"{num_trees:<12} {max_depth:<12} {avg_metric:<12.4f}")

print("\n✅ Business Goal Check:")
print(f"  Recall > 0.70? {'✅ Yes' if recall_best_rf > 0.70 else '❌ No'} (Actual: {recall_best_rf:.4f})")
print(f"  Precision > 0.55? {'✅ Yes' if precision_best_rf > 0.55 else '❌ No'} (Actual: {precision_best_rf:.4f})")
print(f"  AUC-ROC > 0.82? {'✅ Yes' if auc_best_rf > 0.82 else '❌ No'} (Actual: {auc_best_rf:.4f})")

print("\n🎯 CrossValidator found the optimal hyperparameters via systematic search!")
print("   This ensures we didn't overfit to a single validation split.")

---

## Extra Lab (Optional/Async): End-to-End Production Churn Pipeline

**Objective**: Build a complete, production-ready churn prediction pipeline that combines everything you learned this week.

**Challenge**: You're deploying this model to production. Build a pipeline that:
1. Uses the full feature set (all categorical + numerical features in the dataset)
2. Applies proper imbalance handling
3. Uses an optimized ensemble model (Random Forest or GBT)
4. Is tuned via CrossValidator
5. Achieves business goals: Recall > 0.75, Precision > 0.60, AUC > 0.85

**Step-by-Step Guidance**:

1. **Feature Engineering**:
   - Explore the full churn dataset schema
   - Include ALL relevant features: `Contract`, `InternetService`, `PaymentMethod`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaperlessBilling`, `Partner`, `Dependents`, `gender`, `SeniorCitizen`
   - Numerical: `tenure`, `MonthlyCharges`, `TotalCharges`

2. **Advanced Imbalance Handling**:
   - Experiment with SMOTE-like techniques (if time permits) or stick with oversampling
   - Consider stratified sampling to maintain class distribution across folds

3. **Model Selection**:
   - Try BOTH Random Forest and GBT
   - Use CrossValidator to tune:
     - RF: `numTrees`, `maxDepth`, `featureSubsetStrategy`
     - GBT: `maxIter`, `maxDepth`, `stepSize`

4. **Evaluation**:
   - Report all 5 metrics on test set
   - Create confusion matrix (True Positives, False Positives, etc.)
   - Calculate per-class precision and recall (not just weighted average)

5. **Production Considerations**:
   - Save the best model using `model.save("path/to/model")`
   - Document feature importance
   - Create a prediction function that takes raw customer data → churn probability
   - Discuss how you'd monitor model performance in production

**Success Criteria**:
- AUC-ROC > 0.85
- Recall > 0.75 (catching 75%+ of churners)
- Precision > 0.60 (60%+ of flagged customers actually churn)
- Pipeline is reproducible and well-documented

**Hints**:
- Start simple, iterate to complex
- Use smaller CV grid initially (2-3 combinations) to test pipeline end-to-end
- Expand grid once you're confident the pipeline works
- This lab could take 30-60 minutes - perfect for async/homework!

**Real-World Application**:
In production, this model would:
- Run daily/weekly on customer data
- Generate churn risk scores for all active customers
- Feed into CRM system to trigger retention campaigns
- Be monitored for drift (are predictions still accurate 6 months later?)

Good luck! This is the most challenging lab, but also the most rewarding. You're building a real production ML system.